# Calvin Site 1 spatial parameter review

The default loads the QC candidate from the local study. Select `sobol056` to review
the earlier measured-taper starter. These are exploratory inputs. The depth-only
benchmark provides a comparison for the fit across depth.

Run this notebook with the project's Python 3.10 environment. To read the saved study
files from another checkout, set `DEEPROOTGEN_RESEARCH_ROOT` to the local project folder
containing the research bundles and `.runtime`. A clean clone uses the labelled synthetic
example. Save an executed copy under `.runtime` when working with field results.

Run All to generate one seeded realization. For the full saved comparison,
set `run_mode = "confirmation"` in the first code cell and Run All again.

For the 20 November selection, open the [executed parameter record and lessons](Calvin_private/explored_parameters_and_lessons.ipynb) or the [illustrated eight-seed review](Calvin_private/site1_selected_2026-11-20/site1_current_best.ipynb). The candidate below belongs to an earlier comparison.

In [ ]:
from hashlib import sha256
from pathlib import Path
import json
import os
import sys

from IPython.display import Image, display
import pandas as pd

cwd = Path.cwd()
checkout = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "deeprootgen/data_model/canonical.py").is_file()),
    None,
)
if checkout is None:
    raise RuntimeError("Start Jupyter in the DeepRootGen checkout.")
sys.path.insert(0, str(checkout))

from deeprootgen.data_model.resolution import load_json_config, resolve_config
from deeprootgen.run_records import tracked_run

research_root = Path(os.environ.get("DEEPROOTGEN_RESEARCH_ROOT", str(checkout))).expanduser().resolve()
spatial = research_root / "Calvin_SO4_Parameter_Exploration_2026-11-18/Spatial_Evaluation_2026-11-19"

study_id = "calvin-site1"
cases = {
    "qc_refine35": ".runtime/structural-search-20261118/Calvin-qc/Calvin_Site1_qc_start.schema4.private.json",
    "sobol056": "Calvin_SO4_Parameter_Exploration_2026-11-18/Calvin_Site1_start.schema4.private.json",
}
case = "qc_refine35"
synthetic_example = "examples/research/native_cohort_synthetic.json"

run_mode = "preview"  # "review", "preview", or "confirmation"
preview_seed = None   # Keep the saved seed, or enter a different integer for a preview.
confirmation_workers = 2

config_path = research_root / cases[case]
field_case = config_path.is_file()
if not field_case and run_mode == "confirmation":
    raise FileNotFoundError(f"Set DEEPROOTGEN_RESEARCH_ROOT to the local bundle containing {config_path}")
if not field_case:
    config_path = checkout / synthetic_example
    print("Synthetic example. Set DEEPROOTGEN_RESEARCH_ROOT to load the saved study case.")

config_bytes = config_path.read_bytes()
config = load_json_config(config_bytes.decode("utf-8"))
resolved = resolve_config(config)
print(f"{case if field_case else 'synthetic'}  |  schema {config['schema_version']}  |  seed {resolved.seed}")
print("Configuration", config_path)
print("Input file SHA-256", sha256(config_bytes).hexdigest())

## Review the input

The first table lists supplied scalar settings and their resolver metadata. Feature
tables show the order-specific laws. Empirical length arrays are summarized by their
size and bounds, and daughter counts by their conditional range and mean. Open the
referenced JSON for exact values and arrays.

In [ ]:
consumers = resolved.description["consumers"]
parameter_rows = []
for name, value in config["parameters"].items():
    details = consumers.get(name, {})
    parameter_rows.append({
        "parameter": name,
        "value": value,
        "units": details.get("units", ""),
        "operation": details.get("operation", ""),
    })
display(pd.DataFrame(parameter_rows))

order_rows, feature_rows = [], []
for name, feature in config.get("features", {}).items():
    if not feature["enabled"]:
        continue
    settings = feature.get("settings", {})
    other_settings = {k: v for k, v in settings.items() if k != "by_order"}
    feature_rows.append({"feature": name, "settings": other_settings})
    for order, rule in settings.get("by_order", {}).items():
        detail = dict(rule) if isinstance(rule, dict) else {"length_cm": rule}
        if detail.get("law") == "discrete_cdf":
            lengths = detail.pop("lengths_cm")
            detail.pop("cumulative_probabilities")
            detail.update(support_points=len(lengths), min_cm=min(lengths), max_cm=max(lengths))
        if "daughter_probabilities" in detail:
            probabilities = detail.pop("daughter_probabilities")
            supported = [i for i, value in enumerate(probabilities, 1) if value > 0]
            detail.update(daughter_count_min=min(supported, default=None), daughter_count_max=max(supported, default=None),
                          conditional_mean_daughters=sum(i * value for i, value in enumerate(probabilities, 1)))
        order_rows.append({"order": int(order), "feature": name, "settings": detail})

if feature_rows:
    display(pd.DataFrame(feature_rows))
if order_rows:
    display(pd.DataFrame(order_rows).sort_values(["order", "feature"]))

## Matched core comparisons in the saved Calvin study

Compare root length density (RLD) and dry-mass density at the same measured core and depth.
The original spatial evaluation holds out complete Site 1 cores. The reduced support omits
flagged scanner totals. Site 5 is an external diagnostic. Read the paired errors alongside
joint location-depth and depth-only allocation errors.

In [ ]:
scores_path = spatial / "calvin_spatial_scores.csv"
if field_case and scores_path.is_file():
    scores = pd.read_csv(scores_path)
    panel = {"qc_refine35": "Calvin-qc", "sobol056": "combined_28"}[case]
    comparison = scores.panel.eq(panel) & scores.candidate.eq(case)
    baseline = scores.panel.eq("baseline") & scores.candidate.eq("depth_only")
    supports = ["Site1_hold", "Site1_hold_qc", "Site5_external"]
    columns = ["candidate", "fold", "seeds", "rld_nmae", "mass_nmae",
               "rld_joint_tv", "mass_joint_tv", "rld_depth_tv", "mass_depth_tv", "spatial_score"]
    display(scores.loc[(comparison | baseline) & scores.fold.isin(supports), columns])
    figure_path = spatial / "calvin_held_core_profiles.png"
    if figure_path.is_file():
        display(Image(filename=str(figure_path)))
else:
    print("Saved Calvin spatial scores are available with the local study bundle.")

## Run the generator or reproduce the confirmation

**Preview** runs the selected JSON at its saved seed through the current engine. It saves the input,
geometry, material summaries, a 3D view, and a run record. The preview is one realization.

**Confirmation** re-runs 16 reserved seeds for qc_refine35 and 28 for sobol056. It uses the recorded generator and the local study bundle,
verifies the recreated predictions, then recalculates the spatial scores at the original observation support.
The output contains the score table and per-seed receipts. Allow several minutes, especially for Calvin.

**Review** opens the settings and saved comparisons without running the generator.


In [ ]:
from subprocess import run
from uuid import uuid4

run_root = checkout / ".runtime/notebook-review-runs"
label = f"{study_id}-{case if field_case else 'synthetic'}"
run_dir = run_root / f"{label}-{run_mode}-{uuid4().hex[:8]}"
if run_mode == "preview":
    def set_seed(data):
        data["random_seed"] = preview_seed

    record = tracked_run(
        config_path, run_dir, label=label, ledger=run_root / "runs.jsonl",
        edit=set_seed if preview_seed is not None else None,
    )
    summary = json.loads((run_dir / "root_summary.json").read_bytes())
    display(pd.DataFrame(summary["by_order"]))
    display(pd.DataFrame(summary["diameter_classes"]).T)
    print("Saved preview", run_dir)
    print("Run identity", record["model_identity"])
elif run_mode == "confirmation":
    runner = checkout / "notebooks/grapevine_root_architecture/replay_confirmation.py"
    run([
        sys.executable, str(runner), "--research-root", str(research_root), "--study", study_id,
        "--case", case, "--config", str(config_path), "--outdir", str(run_dir),
        "--workers", str(confirmation_workers),
    ], cwd=checkout, check=True)
    display(pd.read_csv(run_dir / "scores.csv"))
    print("Verified confirmation", run_dir / "receipt.json")
elif run_mode != "review":
    raise ValueError(f"Unknown run mode: {run_mode}")
